# 📦 Dataset Downloader for Image Forgery Detection

This notebook downloads and extracts the following datasets directly into your **Google Drive**:

| # | Dataset | Contents | Size |
|---|---------|----------|------|
| 1 | **CASIA 1.0 (Corel)** | 800 authentic + 921 spliced images + Ground Truth masks | ~95 MB |
| 2 | **CocoGlide** | 512 GLIDE-inpainted images + masks (from TruFor) | ~123 MB |
| 3 | **Columbia Uncompressed** | 183 authentic + 180 spliced uncompressed images | ~400 MB |

> 💡 **Features:**
> - Automatically removes corrupted/partial downloads
> - Handles Google Drive virus scan bypass
> - Unpacks both `.zip` and inner `.tar.bz2` archives for Columbia
> - Safe extraction with archive integrity verification

---
## 1️⃣ Mount Google Drive & Setup Helper Functions

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import tarfile
import zipfile
import shutil
import requests
from tqdm.notebook import tqdm

# ============================================================
# 📁 CONFIGURE YOUR BASE DIRECTORY ON GOOGLE DRIVE
# ============================================================
BASE_DIR = "/content/drive/MyDrive/Datasets/ForgeryDetection"
os.makedirs(BASE_DIR, exist_ok=True)
print(f"✅ Target Base Directory: {BASE_DIR}")

def get_dataset_dir(name):
    """Creates and returns a specific dataset folder."""
    path = os.path.join(BASE_DIR, name)
    os.makedirs(path, exist_ok=True)
    return path

def print_dir_stats(path):
    """Counts files and subdirectories."""
    total_files = 0
    total_dirs = 0
    for root, dirs, files in os.walk(path):
        total_dirs += len(dirs)
        total_files += len(files)
    print(f"   📊 Found {total_files} files across {total_dirs} directories in {os.path.basename(path)}.")

def safe_extract(zip_path, extract_to):
    """Safely extracts a zip file with progress output."""
    if not zipfile.is_zipfile(zip_path):
        raise ValueError(f"'{zip_path}' is not a valid zip archive!")
    print(f"📦 Extracting {os.path.basename(zip_path)} to {extract_to}...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_to)
    print("✅ Extraction finished successfully!")

def download_file(url, dest_path, desc=None):
    """Streams a file from a standard direct URL with a progress bar."""
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    response = requests.get(url, headers=headers, stream=True, allow_redirects=True)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    desc = desc or os.path.basename(dest_path)
    
    with open(dest_path, 'wb') as f, tqdm(
        total=total_size, unit='B', unit_scale=True, unit_divisor=1024, desc=desc
    ) as bar:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

def download_gdrive_large(file_id, dest_path, desc=None):
    """
    Downloads large Google Drive files by solving the virus scan confirmation screen.
    """
    session = requests.Session()
    url = f"https://drive.google.com/uc?id={file_id}&export=download"
    res = session.get(url, stream=True)
    
    content_type = res.headers.get("content-type", "")
    if "text/html" in content_type:
        html = res.text
        action_match = re.search(r'action="([^"]+)"', html)
        inputs = dict(re.findall(r'<input[^>]+name="([^"]+)"[^>]+value="([^"]*)"', html))
        if action_match and inputs:
            action_url = action_match.group(1)
            res = session.get(action_url, params=inputs, stream=True)
    
    res.raise_for_status()
    total_size = int(res.headers.get('content-length', 0))
    desc = desc or os.path.basename(dest_path)
    
    with open(dest_path, 'wb') as f, tqdm(
        total=total_size, unit='B', unit_scale=True, unit_divisor=1024, desc=desc
    ) as bar:
        for chunk in res.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

print("✅ Download helper utilities initialized.")

---
## 2️⃣ Download CASIA 1.0 (Corel) & Ground Truth Masks

**Contents:**
- Authentic images (800 total)
- Spliced images (921 total)
- Ground truth binary masks (from namtpham/casia1groundtruth)

In [ ]:
casia_dir = get_dataset_dir("CASIA1.0")
casia_zip = os.path.join(casia_dir, "CASIA1.0_dataset.zip")
CASIA_GDRIVE_FILE_ID = "14f3jU2VsxTYopgSE1Vvv4hMlFvpAKHUY"

# 1. Clean up corrupted or partial zip files
if os.path.exists(casia_zip):
    if not zipfile.is_zipfile(casia_zip):
        print(f"⚠️ Existing '{os.path.basename(casia_zip)}' is corrupt/not a valid zip ({os.path.getsize(casia_zip)} bytes). Deleting it...")
        os.remove(casia_zip)
    else:
        print("⏭️  Valid CASIA 1.0 zip already exists on Drive.")

# 2. Download from Google Drive if needed
if not os.path.exists(casia_zip):
    print("⬇️  Downloading CASIA 1.0 dataset (~95 MB)... ")
    try:
        download_gdrive_large(CASIA_GDRIVE_FILE_ID, casia_zip, desc="CASIA 1.0 Dataset")
    except Exception as e:
        print(f"❌ Download failed: {e}")
        if os.path.exists(casia_zip):
            os.remove(casia_zip)
        raise

# 3. Verify and extract
if os.path.exists(casia_zip) and zipfile.is_zipfile(casia_zip):
    safe_extract(casia_zip, casia_dir)
else:
    print("❌ Could not extract: file is not a valid zip archive.")

# 4. Clone Ground Truth binary masks from official repository
casia_gt_dir = os.path.join(casia_dir, "casia1groundtruth")
if not os.path.exists(casia_gt_dir) or len(os.listdir(casia_gt_dir)) == 0:
    print("\n⬇️  Cloning CASIA 1.0 Ground Truth masks...")
    !git clone https://github.com/namtpham/casia1groundtruth.git "{casia_gt_dir}"
    print("✅ Ground Truth masks cloned successfully!")
else:
    print("⏭️  Ground Truth folder already present, skipping clone.")

print("\n📁 CASIA 1.0 Final Directory Status:")
print_dir_stats(casia_dir)

---
## 3️⃣ Download CocoGlide Dataset

**Source:** University Federico II of Naples (GRIP-UNINA / TruFor)  
**Contents:** 512 manipulated images generated with OpenAI's GLIDE diffusion model + ground truth masks.

In [ ]:
cocoglide_dir = get_dataset_dir("CocoGlide")
cocoglide_zip = os.path.join(cocoglide_dir, "CocoGlide.zip")
COCOGLIDE_URL = "https://www.grip.unina.it/download/prog/TruFor/CocoGlide.zip"

# 1. Clean up invalid zip if any
if os.path.exists(cocoglide_zip) and not zipfile.is_zipfile(cocoglide_zip):
    print(f"⚠️ Existing CocoGlide.zip is invalid. Removing...")
    os.remove(cocoglide_zip)

# 2. Download
if not os.path.exists(cocoglide_zip):
    print("⬇️  Downloading CocoGlide (~123 MB)...")
    download_file(COCOGLIDE_URL, cocoglide_zip, desc="CocoGlide.zip")
else:
    print("⏭️  Valid CocoGlide zip already exists, skipping download.")

# 3. Extract
if os.path.exists(cocoglide_zip) and zipfile.is_zipfile(cocoglide_zip):
    safe_extract(cocoglide_zip, cocoglide_dir)

print("\n📁 CocoGlide Final Directory Status:")
print_dir_stats(cocoglide_dir)

---
## 4️⃣ Download Columbia Uncompressed Splicing Dataset

**Contents:** 183 authentic + 180 spliced uncompressed images captured across 4 cameras with masks.

*Note: The Dropbox archive contains `4cam_auth.tar.bz2` and `4cam_splc.tar.bz2`. This cell unzips the bundle AND unpacks both tar.bz2 archives so all individual image files are extracted.*

In [ ]:
columbia_dir = get_dataset_dir("Columbia")
columbia_zip = os.path.join(columbia_dir, "Columbia_Uncompressed.zip")
COLUMBIA_DROPBOX_URL = "https://www.dropbox.com/sh/786qv3yhvc7s9ki/AACbEEzGPrD3_y38bpWHzgdqa?dl=1"

# 1. Clean up corrupted zip if any
if os.path.exists(columbia_zip) and not zipfile.is_zipfile(columbia_zip):
    print(f"⚠️ Existing Columbia zip is invalid. Removing...")
    os.remove(columbia_zip)

# 2. Download
if not os.path.exists(columbia_zip):
    print("⬇️  Downloading Columbia Uncompressed Dataset (~383 MB)...")
    try:
        download_file(COLUMBIA_DROPBOX_URL, columbia_zip, desc="Columbia Dataset")
    except Exception as e:
        print(f"❌ Automated download failed: {e}")
        raise
else:
    print("⏭️  Valid Columbia zip already exists, skipping download.")

# 3. Extract the outer zip bundle
if os.path.exists(columbia_zip) and zipfile.is_zipfile(columbia_zip):
    safe_extract(columbia_zip, columbia_dir)

# 4. Extract the inner .tar.bz2 archives (where the actual image files are!)
for tar_name in ["4cam_auth.tar.bz2", "4cam_splc.tar.bz2"]:
    tar_path = os.path.join(columbia_dir, tar_name)
    if os.path.exists(tar_path):
        print(f"📦 Unpacking image archive: {tar_name}...")
        with tarfile.open(tar_path, "r:bz2") as tar:
            tar.extractall(columbia_dir)
        print(f"✅ {tar_name} unpacked successfully!")

# 5. Sync/flush Drive listing
print("\n📁 Columbia Final Directory Status:")
print_dir_stats(columbia_dir)
print("\nFolders in Columbia:", os.listdir(columbia_dir))
print("Folders in ForgeryDetection:", os.listdir(BASE_DIR))

---
## 5️⃣ Verification & Paths Summary

In [ ]:
print("=" * 75)
print("📊 DATASET STORAGE SUMMARY")
print("=" * 75)
print(f"Base Storage Location on Google Drive: {BASE_DIR}\n")

checks = [
    ("CASIA1.0", os.path.join(BASE_DIR, "CASIA1.0"), 500),
    ("CocoGlide", os.path.join(BASE_DIR, "CocoGlide"), 400),
    ("Columbia", os.path.join(BASE_DIR, "Columbia"), 300),
]

for name, path, min_expected in checks:
    if os.path.exists(path):
        total_files = sum(len(files) for _, _, files in os.walk(path))
        total_size_mb = sum(
            os.path.getsize(os.path.join(root, f))
            for root, _, files in os.walk(path)
            for f in files
        ) / (1024 * 1024)
        
        icon = "✅" if total_files >= min_expected else "⚠️"
        print(f"{icon} {name:<12} | {total_files:>5} files | {total_size_mb:>8.1f} MB | {path}")
    else:
        print(f"❌ {name:<12} | Not found at {path}")

print("=" * 75)
print("\n🎉 How to use in your Colab code:")
print(f'casia_path     = "{BASE_DIR}/CASIA1.0"')
print(f'cocoglide_path = "{BASE_DIR}/CocoGlide"')
print(f'columbia_path  = "{BASE_DIR}/Columbia"')

---
## 🗑️ Optional: Delete Zip & Tar Files to Save Drive Space

Run this cell only after the datasets have been extracted and verified above.

In [ ]:
# Uncomment and run to delete original archives and keep only extracted images
# for root, _, files in os.walk(BASE_DIR):
#     for f in files:
#         if f.endswith((".zip", ".tar.bz2", ".tar.gz", ".tar")):
#             fpath = os.path.join(root, f)
#             size_mb = os.path.getsize(fpath) / (1024 * 1024)
#             print(f"🗑️ Removing {f} ({size_mb:.1f} MB)")
#             os.remove(fpath)
# print("\n✅ Archive cleanup complete.")

---
## 6️⃣ Platform Compression via Simulator (WhatsApp, Instagram, Facebook, Telegram)

This cell applies the exact empirical compression pipelines (quantization tables, resolutions, and codecs) to all datasets.

**Hierarchy Created on Google Drive:**
```
MyDrive/Datasets/ForgeryDetection/
├── CocoGlide/
│   ├── whatsapp/      (real/, fake/)
│   ├── instagram/     (real/, fake/)
│   ├── facebook/      (real/, fake/)
│   ├── telegram/      (real/, fake/)
│   └── mask/          (ground truth binary masks)
├── CASIA1.0/
│   ├── whatsapp/      (Au/, Modified Tp/CM/, Modified Tp/Sp/)
│   ├── instagram/     (Au/, Modified Tp/CM/, Modified Tp/Sp/)
│   ├── facebook/      (Au/, Modified Tp/CM/, Modified Tp/Sp/)
│   └── telegram/      (Au/, Modified Tp/CM/, Modified Tp/Sp/)
└── Columbia/
    ├── whatsapp/      (4cam_auth/, 4cam_splc/)
    ├── instagram/     (4cam_auth/, 4cam_splc/)
    ├── facebook/      (4cam_auth/, 4cam_splc/)
    └── telegram/      (4cam_auth/, 4cam_splc/)
```

In [ ]:
import os
import io
import csv
import time
import shutil
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageOps

# ============================================================================
# EMPIRICAL QUANTIZATION TABLES & PLATFORM PROFILES (FROM SIMULATOR ENGINE)
# ============================================================================
WHATSAPP_STANDARD_QTABLES = {
    0: [6, 6, 6, 7, 10, 15, 22, 34, 6, 7, 8, 11, 14, 16, 21, 30, 6, 8, 10, 12, 17, 25, 36, 54, 7, 11, 12, 16, 21, 30, 42, 62, 10, 14, 17, 21, 28, 38, 52, 76, 15, 16, 25, 30, 38, 50, 68, 95, 22, 21, 36, 42, 52, 68, 90, 124, 34, 30, 54, 62, 76, 95, 124, 167],
    1: [6, 6, 6, 7, 10, 15, 22, 34, 6, 7, 8, 11, 14, 16, 21, 30, 6, 8, 10, 12, 17, 25, 36, 54, 7, 11, 12, 16, 21, 30, 42, 62, 10, 14, 17, 21, 28, 38, 52, 76, 15, 16, 25, 30, 38, 50, 68, 95, 22, 21, 36, 42, 52, 68, 90, 124, 34, 30, 54, 62, 76, 95, 124, 167]
}
WHATSAPP_HD_QTABLES = {
    0: [3, 2, 2, 3, 4, 6, 8, 10, 2, 2, 2, 3, 4, 9, 10, 9, 2, 2, 3, 4, 6, 9, 11, 9, 2, 3, 4, 5, 8, 14, 13, 10, 3, 4, 6, 9, 11, 17, 16, 12, 4, 6, 9, 10, 13, 17, 18, 15, 8, 10, 12, 14, 16, 19, 19, 16, 12, 15, 15, 16, 18, 16, 16, 16],
    1: [3, 3, 4, 8, 16, 16, 16, 16, 3, 3, 4, 11, 16, 16, 16, 16, 4, 4, 9, 16, 16, 16, 16, 16, 8, 11, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16]
}
TELEGRAM_QTABLES = {
    0: [4, 3, 3, 4, 6, 10, 13, 16, 3, 3, 4, 5, 7, 15, 16, 14, 4, 3, 4, 6, 10, 15, 18, 15, 4, 4, 6, 8, 13, 23, 21, 16, 5, 6, 10, 15, 18, 28, 27, 20, 6, 9, 14, 17, 21, 27, 29, 24, 13, 17, 20, 23, 27, 31, 31, 26, 19, 24, 25, 25, 29, 26, 27, 26],
    1: [4, 5, 6, 12, 26, 26, 26, 26, 5, 5, 7, 17, 26, 26, 26, 26, 6, 7, 15, 26, 26, 26, 26, 26, 12, 17, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26]
}
FACEBOOK_QTABLES = {
    0: [9, 8, 16, 16, 23, 24, 26, 27, 8, 11, 15, 17, 24, 22, 28, 28, 16, 15, 18, 20, 24, 27, 26, 29, 16, 17, 20, 23, 29, 31, 31, 32, 23, 24, 24, 29, 32, 34, 38, 38, 24, 22, 27, 31, 34, 38, 35, 42, 26, 28, 26, 31, 38, 35, 38, 37, 27, 28, 29, 32, 38, 42, 37, 30],
    1: [8, 7, 14, 12, 19, 17, 20, 22, 7, 10, 13, 13, 17, 15, 19, 20, 14, 13, 15, 14, 16, 18, 18, 21, 12, 13, 14, 15, 19, 17, 19, 22, 19, 17, 16, 19, 15, 22, 17, 21, 17, 15, 18, 17, 22, 19, 22, 32, 20, 19, 18, 19, 17, 22, 35, 32, 22, 20, 21, 22, 21, 32, 32, 255]
}
PROFILES = {
    'whatsapp': {'codec': 'JPEG', 'max_dimension': 1600, 'adaptive_whatsapp': True, 'optimize': True, 'progressive': True},
    'instagram': {'codec': 'WEBP', 'max_dimension': 1440, 'quality': 65},
    'facebook': {'codec': 'JPEG', 'max_dimension': 1080, 'qtables': FACEBOOK_QTABLES, 'optimize': True},
    'telegram': {'codec': 'JPEG', 'max_dimension': 1280, 'qtables': TELEGRAM_QTABLES, 'optimize': True},
}

def prepare_img(img):
    img = ImageOps.exif_transpose(img)
    if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
        rgba = img.convert('RGBA')
        bg = Image.new('RGB', rgba.size, (255, 255, 255))
        bg.paste(rgba, mask=rgba.split()[3])
        return bg
    return img.convert('RGB')

def sim_compress(img, profile):
    img = prepare_img(img)
    orig_w, orig_h = img.size
    max_dim = profile['max_dimension']
    if max(orig_w, orig_h) > max_dim:
        scale = max_dim / max(orig_w, orig_h)
        img = img.resize((int(orig_w * scale), int(orig_h * scale)), Image.LANCZOS)
    buf = io.BytesIO()
    codec = profile['codec']
    if codec == 'JPEG':
        if profile.get('adaptive_whatsapp'):
            qtables = WHATSAPP_STANDARD_QTABLES if max(orig_w, orig_h) > 800 else WHATSAPP_HD_QTABLES
        else:
            qtables = profile.get('qtables')
        img.save(buf, format='JPEG', qtables=qtables, subsampling=2, optimize=True, progressive=profile.get('progressive', False))
    else:
        img.save(buf, format='WEBP', quality=profile.get('quality', 65), method=4)
    return buf.getvalue(), codec

# ============================================================================
# BATCH DATASET PROCESSOR
# ============================================================================
def run_compression_pipeline(base_dir):
    image_exts = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.webp', '.bmp')
    platforms = ['whatsapp', 'instagram', 'facebook', 'telegram']
    
    tasks = {'CocoGlide': [], 'CASIA1.0': [], 'Columbia': []}
    
    # 1. Map CocoGlide
    cg_dir = os.path.join(base_dir, 'CocoGlide')
    if os.path.exists(cg_dir):
        for s in ['real', 'fake']:
            sd = os.path.join(cg_dir, s)
            if os.path.exists(sd):
                for f in sorted(os.listdir(sd)):
                    if f.lower().endswith(image_exts):
                        tasks['CocoGlide'].append((os.path.join(sd, f), s))
    
    # 2. Map CASIA 1.0
    casia_dir = os.path.join(base_dir, 'CASIA1.0')
    if os.path.exists(casia_dir):
        for root, _, files in os.walk(casia_dir):
            for f in files:
                if f.lower().endswith(image_exts):
                    fp = os.path.join(root, f)
                    if 'casia1groundtruth' in fp:
                        continue
                    if 'Au' in fp and 'Modified' not in fp and 'Original' not in fp:
                        tasks['CASIA1.0'].append((fp, 'Au'))
                    elif 'Modified Tp' in fp or 'Modified_CopyMove' in fp or ('Tp' in fp and 'CM' in fp):
                        tasks['CASIA1.0'].append((fp, 'Modified Tp/CM'))
                    elif 'Modified' in fp or ('Tp' in fp and 'Sp' in fp):
                        tasks['CASIA1.0'].append((fp, 'Modified Tp/Sp'))
    
    # 3. Map Columbia
    col_dir = os.path.join(base_dir, 'Columbia')
    if os.path.exists(col_dir):
        for s in ['4cam_auth', '4cam_splc']:
            sd = os.path.join(col_dir, s)
            if os.path.exists(sd):
                for f in sorted(os.listdir(sd)):
                    fp = os.path.join(sd, f)
                    if os.path.isfile(fp) and f.lower().endswith(image_exts):
                        tasks['Columbia'].append((fp, s))
                        
    total_src = sum(len(v) for v in tasks.values())
    print(f'Discovered {total_src} unique source images.')
    print(f'Generating {total_src * len(platforms)} platform images across {platforms}...\n')
    
    def worker(item):
        src, dst, plat = item
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        with Image.open(src) as im:
            b, codec = sim_compress(im, PROFILES[plat])
        with open(dst, 'wb') as f:
            f.write(b)
        return len(b)
        
    for ds_name, items in tasks.items():
        if not items:
            continue
        print(f'🔄 Processing {ds_name} ({len(items)} images)...')
        t0 = time.time()
        work = []
        for src, sub in items:
            stem = os.path.splitext(os.path.basename(src))[0]
            for plat in platforms:
                ext = '.webp' if PROFILES[plat]['codec'] == 'WEBP' else '.jpg'
                dst = os.path.join(base_dir, ds_name, plat, sub, f'{stem}{ext}')
                work.append((src, dst, plat))
        with ThreadPoolExecutor(max_workers=8) as ex:
            list(ex.map(worker, work))
        print(f'   ✅ {ds_name} done: {len(work)} images in {time.time()-t0:.1f}s')
        
    print('\n🎉 All platform datasets created successfully in:', base_dir)

run_compression_pipeline(BASE_DIR)